In [0]:
import os
import re
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import col
from pyspark.sql.types import StringType

CLEAR_RAW = False # True or False

def pascal_to_upper_snake(name: str) -> str:
    # Insere "_" antes de letras maiúsculas (exceto na primeira posição) e converte tudo para UPPERCASE
    return re.sub(r'(?<!^)(?=[A-Z])', '_', name).upper()

def drop_tables(tables, schema="adventure_works.bronze"):
    for name in tables:
        query = f"DROP TABLE IF EXISTS {schema}.{name}"
        spark.sql(query)

# api
dir_api = "/Volumes/adventure_works/raw/raw_api"


files_api = [file for file in os.listdir(dir_api)]
tables_api = [pascal_to_upper_snake(file.split('.')[0]) for file in files_api]
dir_tables_api = [dir_api+'/'+file for file in files_api]

# db
dir_db = "/Volumes/adventure_works/raw/raw_db"


files_db = [file for file in os.listdir(dir_db)]
tables_db = [pascal_to_upper_snake(file.split('.')[0]) for file in files_db]
dir_tables_db = [f"{dir_db}/{file}" for file in files_db]

all_tables = tables_api + tables_db
list_to_delta = dir_tables_api + dir_tables_db
files = files_api + files_db

if CLEAR_RAW:
    drop_tables(all_tables)

for file, abs_path in zip(files, list_to_delta):

    # if 'ProductVendor.jsonl' in file:

    df_bronze = (
        spark.read
        .format("json")
        .option("multiLine", "false")
        .load(abs_path)
        )
    
    df_bronze = df_bronze.withColumn("load_at", current_timestamp())

    df_bronze = df_bronze.select([col(c).cast(StringType()).alias(c) for c in df_bronze.columns])

    save_delta_name = pascal_to_upper_snake(file.split('.')[0])
    print(save_delta_name.upper())

    # Escrita no formato Delta para a camada Bronze
    df_bronze.write\
        .option("overwriteSchema", "true")\
        .format("delta")\
        .mode("append")\
        .saveAsTable(f"adventure_works.bronze.{save_delta_name.upper()}")
